In [1]:
!pip -q install scikit-learn pandas opendatasets

In [2]:
import opendatasets as od

dataset_url = 'https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews'
od.download(dataset_url)

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews


100%|██████████| 25.7M/25.7M [00:00<00:00, 1.45GB/s]

In [3]:
import pandas as pd

df = pd.read_csv("/content/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
df = df.sample(10000).reset_index(drop=True)
print(len(df))
df.head()

10000


,review,sentiment
0,Alfred Hitchcock has made many brilliant thril...,positive
1,i completely agree with jamrom4.. this was the...,negative
2,This is a film that has to be taken in context...,positive
3,I am a Motion Picture Production major at Wrig...,negative
4,Tom is about to tuck into a delicious Jerry sa...,negative


In [4]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
0,Alfred Hitchcock has made many brilliant thril...,1
1,i completely agree with jamrom4.. this was the...,0
2,This is a film that has to be taken in context...,1
3,I am a Motion Picture Production major at Wrig...,0
4,Tom is about to tuck into a delicious Jerry sa...,0


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [10]:
from sklearn.svm import SVC

model = SVC(kernel='linear', probability=True)
model.fit(X_train_tfidf, y_train)

SVC(kernel='linear', probability=True)

In [12]:
y_pred = model.predict(X_test_tfidf)

In [13]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8605


In [14]:
def predict_sentiment(text):

    # Convert text to TF-IDF features
    text_tfidf = vectorizer.transform([text])

    # Predict sentiment
    prediction = model.predict(text_tfidf)[0]

    # Get probabilities
    probs = model.predict_proba(text_tfidf)[0]

    positive_prob = probs[1]
    negative_prob = probs[0]

    sentiment = "Positive" if prediction == 1 else "Negative"

    return sentiment, positive_prob, negative_prob

In [25]:
sentiment, pos_prob, neg_prob = predict_sentiment("This movie is incredibly boring")

print("Sentiment:", sentiment)
print("Positive Probability:", round(pos_prob, 2))
print("Negative Probability:", round(neg_prob, 2))

Sentiment: Negative
Positive Probability: 0.03
Negative Probability: 0.97


In [29]:
import pickle

# Save SVM model
with open("svm_sentiment_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)